# 06 - Interpretación de Modelos

**Pregunta de negocio:** ¿Podemos explicar las predicciones al negocio?

## Objetivos
- Interpretar los modelos de clasificación y regresión entrenados en notebooks anteriores
- Aplicar técnicas de interpretación global (permutation importance, PDPs) y local (predicciones individuales)
- Crear "model cards" con información clave para stakeholders
- Generar insights accionables que el negocio pueda usar

## Teoría: ¿Por Qué Importa la Interpretación?

### Motivaciones
1. **Confianza**: los stakeholders no confiarán en un modelo que no pueden entender
2. **Debugging**: detectar si el modelo usa features espurias o tiene data leakage
3. **Cumplimiento regulatorio**: en muchos sectores (finanzas, salud) es obligatorio explicar las decisiones
4. **Mejora continua**: entender qué aprende el modelo para mejorar features y datos

### Interpretación Global vs Local
- **Global**: ¿qué features son importantes *en general*? (permutation importance, PDPs)
- **Local**: ¿por qué el modelo hizo *esta* predicción específica? (análisis de casos individuales)

### Técnicas
- **Permutation Importance**: permutar aleatoriamente una feature y medir cuánto empeora el modelo
  - Ventaja: model-agnostic, funciona con cualquier modelo
  - Ventaja: mide la importancia en el set de test (no se sobreajusta)
- **Feature Importance (built-in)**: importancia basada en Gini (RF) o gain (XGBoost)
  - Ventaja: rápido, integrado en el modelo
  - Desventaja: puede sobrestimar features con muchas categorías
- **Partial Dependence Plots (PDPs)**: muestran el efecto marginal de una feature en la predicción
  - ¿Cómo cambia la predicción si variamos *solo* esta feature?
  - Asume independencia entre features (limitación)

In [ ]:
import os
import glob
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import (
    classification_report, roc_auc_score, f1_score,
    mean_squared_error, r2_score, mean_absolute_error
)

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier as XGBClassifier
    HAS_XGBOOST = False

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
data_dir = os.path.join(project_root, "data/raw")
processed_dir = os.path.join(project_root, "data/processed")
models_dir = os.path.join(project_root, "models")
rng = np.random.default_rng(42)

vtype_colors = {'electrico': '#2ecc71', 'gasolina': '#3498db', 'hibrido': '#9b59b6', 'deportivo': '#e74c3c'}

print("Librerías cargadas correctamente")

## 1. Cargar modelos guardados

Cargamos los modelos de los notebooks anteriores. Si no existen, los reentrenamos.

In [ ]:
# Intentar cargar modelos guardados
risk_model_path = os.path.join(models_dir, "classification_driver_risk.pkl")
churn_model_path = os.path.join(models_dir, "classification_churn_risk.pkl")

# --- Modelo de riesgo de conductor ---
if os.path.exists(risk_model_path):
    risk_pkg = joblib.load(risk_model_path)
    risk_model = risk_pkg['model']
    risk_features = risk_pkg['feature_cols']
    print(f"Modelo de riesgo cargado: {risk_pkg['model_name']}")
    print(f"  Features: {risk_features}")
else:
    print("Modelo de riesgo no encontrado, reentrenando...")
    risk_pkg = None

# --- Modelo de churn ---
if os.path.exists(churn_model_path):
    churn_pkg = joblib.load(churn_model_path)
    churn_model = churn_pkg['model']
    churn_features = churn_pkg['feature_cols']
    print(f"\nModelo de churn cargado: {churn_pkg['model_name']}")
    print(f"  Features: {churn_features}")
else:
    print("\nModelo de churn no encontrado, reentrenando...")
    churn_pkg = None

In [ ]:
# Cargar datos y preparar para análisis
# --- Datos de riesgo (telemetría) ---
features_path = os.path.join(processed_dir, "features_ml.csv")
if os.path.exists(features_path):
    df_risk = pd.read_csv(features_path)
else:
    files = sorted(glob.glob(os.path.join(data_dir, "telemetry/telemetry_*.csv")))
    telemetry = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    fleet = pd.read_csv(os.path.join(data_dir, "fleet_profiles.csv"))
    telemetry = telemetry.merge(fleet[['vehicle_id', 'vehicle_type']], on='vehicle_id', how='left')
    df_risk = telemetry.groupby('vehicle_id').agg(
        speed_mean=('speed_kmh', 'mean'), speed_std=('speed_kmh', 'std'),
        speed_max=('speed_kmh', 'max'), consumption_mean=('fuel_consumption_rate', 'mean'),
        battery_temp_mean=('battery_temp_c', 'mean'), battery_soc_mean=('battery_soc_pct', 'mean'),
        motor_rpm_mean=('motor_rpm', 'mean'), motor_power_mean=('motor_power_kw', 'mean'),
        acceleration_mean=('acceleration_ms2', 'mean'), acceleration_std=('acceleration_ms2', 'std'),
        n_trips=('trip_id', 'nunique'), vehicle_type=('vehicle_type', 'first'),
    ).reset_index()
    harsh = telemetry[telemetry['acceleration_ms2'] < -3].groupby('vehicle_id').size().reset_index(name='harsh_braking_count')
    df_risk = df_risk.merge(harsh, on='vehicle_id', how='left')
    df_risk['harsh_braking_count'] = df_risk['harsh_braking_count'].fillna(0)

print(f"Datos riesgo: {df_risk.shape}")

# --- Datos de churn (encuestas + merged) ---
merged_path = os.path.join(processed_dir, "vehicle_survey_merged.csv")
if os.path.exists(merged_path):
    df_churn = pd.read_csv(merged_path)
else:
    df_churn = pd.read_csv(os.path.join(data_dir, "surveys/buyer_surveys.csv"))

print(f"Datos churn: {df_churn.shape}")

In [ ]:
# Recrear targets y preparar datasets si es necesario

# --- Riesgo de conductor ---
risk_cols_candidates = ['harsh_braking_count', 'speed_std', 'acceleration_std']
risk_cols = [c for c in risk_cols_candidates if c in df_risk.columns][:2]

for col in risk_cols:
    threshold = df_risk[col].quantile(0.80)
    df_risk[f'{col}_high'] = (df_risk[col] >= threshold).astype(int)

if len(risk_cols) >= 2:
    df_risk['aggressive_driver'] = (df_risk[f'{risk_cols[0]}_high'] & df_risk[f'{risk_cols[1]}_high']).astype(int)
else:
    df_risk['aggressive_driver'] = df_risk[f'{risk_cols[0]}_high']

# Preparar features de riesgo
exclude_risk = ['vehicle_id', 'trip_id', 'aggressive_driver'] + risk_cols + [f'{c}_high' for c in risk_cols]
cat_cols_risk = [c for c in df_risk.select_dtypes(include='object').columns if c not in exclude_risk]
df_risk_enc = df_risk.copy()
le_risk = {}
for col in cat_cols_risk:
    le = LabelEncoder()
    df_risk_enc[col] = le.fit_transform(df_risk_enc[col].astype(str))
    le_risk[col] = le

if risk_pkg is not None:
    feat_risk = [c for c in risk_features if c in df_risk_enc.columns]
else:
    feat_risk = [c for c in df_risk_enc.columns if c not in exclude_risk]

X_risk = df_risk_enc[feat_risk].fillna(0)
y_risk = df_risk_enc['aggressive_driver']
X_risk_train, X_risk_test, y_risk_train, y_risk_test = train_test_split(
    X_risk, y_risk, test_size=0.2, random_state=42, stratify=y_risk)

# Reentrenar si no hay modelo guardado
if risk_pkg is None:
    risk_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42,
                                        class_weight='balanced', n_jobs=-1)
    risk_model.fit(X_risk_train, y_risk_train)
    risk_pkg = {'model': risk_model, 'model_name': 'Random Forest (reentrenado)', 'feature_cols': feat_risk}
    print("Modelo de riesgo reentrenado")

# --- Churn ---
df_churn['churn_risk'] = ((df_churn['satisfaction_score'] <= 2) & (df_churn['would_recommend'] == 0)).astype(int)

exclude_churn = ['churn_risk', 'satisfaction_score', 'would_recommend',
                 'vehicle_id', 'trip_id', 'customer_id', 'survey_id', 'respondent_id']
cat_cols_churn = [c for c in df_churn.select_dtypes(include='object').columns if c not in exclude_churn]
df_churn_enc = df_churn.copy()
le_churn = {}
for col in cat_cols_churn:
    le = LabelEncoder()
    df_churn_enc[col] = le.fit_transform(df_churn_enc[col].astype(str))
    le_churn[col] = le

if churn_pkg is not None:
    feat_churn = [c for c in churn_features if c in df_churn_enc.columns]
else:
    num_cols_churn = [c for c in df_churn_enc.select_dtypes(include=[np.number]).columns if c not in exclude_churn]
    feat_churn = [c for c in df_churn_enc.columns if c in num_cols_churn or c in cat_cols_churn]

X_churn = df_churn_enc[feat_churn].fillna(0)
y_churn = df_churn_enc['churn_risk']
X_churn_train, X_churn_test, y_churn_train, y_churn_test = train_test_split(
    X_churn, y_churn, test_size=0.2, random_state=42, stratify=y_churn)

if churn_pkg is None:
    churn_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42,
                                         class_weight='balanced', n_jobs=-1)
    churn_model.fit(X_churn_train, y_churn_train)
    churn_pkg = {'model': churn_model, 'model_name': 'Random Forest (reentrenado)', 'feature_cols': feat_churn}
    print("Modelo de churn reentrenado")

print(f"\nModelo riesgo: {risk_pkg['model_name']} ({len(feat_risk)} features)")
print(f"Modelo churn:  {churn_pkg['model_name']} ({len(feat_churn)} features)")

## 2. Permutation Importance

A diferencia de la importancia built-in (Gini/Gain), la **permutation importance**:
- Se calcula en el set de **test** → no se sobreajusta
- Es **model-agnostic** → funciona con cualquier modelo
- Mide el impacto real de cada feature en la predicción

Método: permutar aleatoriamente los valores de una feature y medir cuánto empeora el score.

In [ ]:
# Permutation importance para ambos modelos
risk_model_obj = risk_pkg['model']
churn_model_obj = churn_pkg['model']

# Riesgo de conductor
perm_risk = permutation_importance(
    risk_model_obj, X_risk_test, y_risk_test,
    n_repeats=10, random_state=42, scoring='f1', n_jobs=-1
)

# Churn
perm_churn = permutation_importance(
    churn_model_obj, X_churn_test, y_churn_test,
    n_repeats=10, random_state=42, scoring='f1', n_jobs=-1
)

print("Permutation importance calculada para ambos modelos")

In [ ]:
# Comparar: Built-in importance vs Permutation importance
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# --- RIESGO: Built-in ---
ax = axes[0, 0]
builtin_risk = pd.DataFrame({
    'Feature': feat_risk,
    'Importancia': risk_model_obj.feature_importances_
}).sort_values('Importancia', ascending=True)
colors_r = plt.cm.Blues(np.linspace(0.3, 0.9, len(builtin_risk)))
ax.barh(builtin_risk['Feature'], builtin_risk['Importancia'], color=colors_r)
ax.set_xlabel('Importancia (Gini/Gain)')
ax.set_title('Riesgo: Built-in Importance', fontsize=13, fontweight='bold')

# --- RIESGO: Permutation ---
ax = axes[0, 1]
perm_risk_df = pd.DataFrame({
    'Feature': feat_risk,
    'Importancia': perm_risk.importances_mean,
    'Std': perm_risk.importances_std
}).sort_values('Importancia', ascending=True)
colors_p = plt.cm.Oranges(np.linspace(0.3, 0.9, len(perm_risk_df)))
ax.barh(perm_risk_df['Feature'], perm_risk_df['Importancia'], 
        xerr=perm_risk_df['Std'], color=colors_p, capsize=3)
ax.set_xlabel('Caída en F1 al permutar')
ax.set_title('Riesgo: Permutation Importance', fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=0.5)

# --- CHURN: Built-in ---
ax = axes[1, 0]
builtin_churn = pd.DataFrame({
    'Feature': feat_churn,
    'Importancia': churn_model_obj.feature_importances_
}).sort_values('Importancia', ascending=True)
colors_c = plt.cm.Greens(np.linspace(0.3, 0.9, len(builtin_churn)))
ax.barh(builtin_churn['Feature'], builtin_churn['Importancia'], color=colors_c)
ax.set_xlabel('Importancia (Gini/Gain)')
ax.set_title('Churn: Built-in Importance', fontsize=13, fontweight='bold')

# --- CHURN: Permutation ---
ax = axes[1, 1]
perm_churn_df = pd.DataFrame({
    'Feature': feat_churn,
    'Importancia': perm_churn.importances_mean,
    'Std': perm_churn.importances_std
}).sort_values('Importancia', ascending=True)
colors_pc = plt.cm.Purples(np.linspace(0.3, 0.9, len(perm_churn_df)))
ax.barh(perm_churn_df['Feature'], perm_churn_df['Importancia'],
        xerr=perm_churn_df['Std'], color=colors_pc, capsize=3)
ax.set_xlabel('Caída en F1 al permutar')
ax.set_title('Churn: Permutation Importance', fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=0.5)

plt.suptitle('Built-in vs Permutation Importance\n(izquierda: importancia del modelo, derecha: impacto real en test)',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("→ Si ambas coinciden → la feature es genuinamente importante")
print("→ Si built-in es alta pero permutation es baja → posible sobreajuste a esa feature")
print("→ Features con importancia negativa en permutation → ruido, eliminar")

In [ ]:
# Concordancia entre rankings
for model_name, builtin_df, perm_df in [
    ('Riesgo', builtin_risk, perm_risk_df),
    ('Churn', builtin_churn, perm_churn_df)
]:
    top5_builtin = set(builtin_df.tail(5)['Feature'].values)
    top5_perm = set(perm_df.tail(5)['Feature'].values)
    overlap = top5_builtin & top5_perm
    
    print(f"\n{model_name}:")
    print(f"  Top 5 built-in:    {list(builtin_df.tail(5)['Feature'].values[::-1])}")
    print(f"  Top 5 permutation: {list(perm_df.tail(5)['Feature'].values[::-1])}")
    print(f"  Coinciden:         {list(overlap)} ({len(overlap)}/5)")

## 3. Partial Dependence Plots (PDPs)

Los PDPs muestran el **efecto marginal** de una feature en la predicción.
Responden: "Si cambio *solo* esta variable, ¿cómo cambia la predicción promedio?"

Muy útiles para comunicar al negocio *cómo* influye cada variable.

In [ ]:
# PDPs para el modelo de riesgo: top 4 features
top4_risk = perm_risk_df.tail(4)['Feature'].values[::-1].tolist()
top4_risk_idx = [feat_risk.index(f) for f in top4_risk if f in feat_risk]

if len(top4_risk_idx) >= 2:
    fig, axes = plt.subplots(1, len(top4_risk_idx), figsize=(5 * len(top4_risk_idx), 5))
    
    display_risk = PartialDependenceDisplay.from_estimator(
        risk_model_obj, X_risk_test, top4_risk_idx,
        kind='average', ax=axes,
        feature_names=feat_risk,
        grid_resolution=50
    )
    
    plt.suptitle('Modelo de Riesgo: Partial Dependence Plots\n(efecto marginal de cada feature en P(agresivo))',
                 fontsize=14, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()
    
    print("→ Pendiente positiva = más de esa variable → más probabilidad de ser agresivo")
    print("→ Pendiente negativa = más de esa variable → menos riesgo")
    print("→ Curva plana = la variable no tiene efecto marginal")
else:
    print("No hay suficientes features para generar PDPs del modelo de riesgo")

In [ ]:
# PDPs para el modelo de churn: top 4 features
top4_churn = perm_churn_df.tail(4)['Feature'].values[::-1].tolist()
top4_churn_idx = [feat_churn.index(f) for f in top4_churn if f in feat_churn]

if len(top4_churn_idx) >= 2:
    fig, axes = plt.subplots(1, len(top4_churn_idx), figsize=(5 * len(top4_churn_idx), 5))
    
    display_churn = PartialDependenceDisplay.from_estimator(
        churn_model_obj, X_churn_test, top4_churn_idx,
        kind='average', ax=axes,
        feature_names=feat_churn,
        grid_resolution=50
    )
    
    plt.suptitle('Modelo de Churn: Partial Dependence Plots\n(efecto marginal de cada feature en P(churn))',
                 fontsize=14, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()
    
    print("→ Cada gráfico muestra cómo cambia la predicción de churn al variar esa feature")
    print("→ Útil para identificar umbrales críticos (ej. 'por debajo de X, el churn se dispara')")
else:
    print("No hay suficientes features para generar PDPs del modelo de churn")

In [ ]:
# PDP 2D: interacción entre las 2 features más importantes (riesgo)
if len(top4_risk_idx) >= 2:
    fig, ax = plt.subplots(figsize=(10, 8))
    
    display_2d = PartialDependenceDisplay.from_estimator(
        risk_model_obj, X_risk_test, 
        [(top4_risk_idx[0], top4_risk_idx[1])],
        kind='average', ax=ax,
        feature_names=feat_risk,
        grid_resolution=30
    )
    
    ax.set_title(f'PDP 2D: Interacción {top4_risk[0]} x {top4_risk[1]}\n'
                 f'(colores = P(agresivo))', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"→ El PDP 2D muestra cómo interactúan {top4_risk[0]} y {top4_risk[1]}")
    print("→ Si el efecto de una feature cambia según el valor de la otra → hay interacción")
    print("→ Colores más oscuros = mayor probabilidad de conductor agresivo")
else:
    print("No hay suficientes features para PDP 2D")

## 4. Predicciones individuales

Seleccionamos 3 vehículos representativos para explicar *por qué* el modelo hizo cada predicción:
1. Un vehículo de **bajo riesgo** (predicción ~0)
2. Un vehículo de **alto riesgo** (predicción ~1)
3. Un caso **borderline** (predicción ~0.5)

In [ ]:
# Obtener probabilidades para seleccionar casos representativos
probs_risk = risk_model_obj.predict_proba(X_risk_test)[:, 1]

# Seleccionar: bajo riesgo, alto riesgo, borderline
idx_low = np.argmin(probs_risk)
idx_high = np.argmax(probs_risk)
idx_border = np.argmin(np.abs(probs_risk - 0.5))

cases = {
    'Bajo riesgo': (idx_low, probs_risk[idx_low]),
    'Alto riesgo': (idx_high, probs_risk[idx_high]),
    'Borderline': (idx_border, probs_risk[idx_border]),
}

print("Casos seleccionados para análisis individual:\n")
for label, (idx, prob) in cases.items():
    actual = y_risk_test.iloc[idx]
    print(f"  {label}: P(agresivo) = {prob:.4f}, Real = {actual}")

In [ ]:
# Análisis detallado de cada caso
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

case_colors = {'Bajo riesgo': '#2ecc71', 'Alto riesgo': '#e74c3c', 'Borderline': '#f39c12'}

# Obtener estadísticas globales para comparar
global_means = X_risk_test.mean()
global_stds = X_risk_test.std()

for ax, (label, (idx, prob)) in zip(axes, cases.items()):
    # Valores del caso
    case_values = X_risk_test.iloc[idx]
    
    # Normalizar respecto al promedio global (z-scores)
    z_scores = (case_values - global_means) / global_stds.replace(0, 1)
    z_df = pd.DataFrame({
        'Feature': feat_risk,
        'Z-score': z_scores.values
    }).sort_values('Z-score')
    
    # Colores por dirección
    bar_colors = ['#e74c3c' if z > 0 else '#3498db' for z in z_df['Z-score']]
    
    ax.barh(z_df['Feature'], z_df['Z-score'], color=bar_colors, alpha=0.8)
    ax.axvline(0, color='black', lw=0.5)
    ax.set_xlabel('Z-score (desviaciones del promedio)')
    ax.set_title(f'{label}\nP(agresivo) = {prob:.3f}\nReal: {"Agresivo" if y_risk_test.iloc[idx] == 1 else "No agresivo"}',
                fontsize=12, fontweight='bold', color=case_colors[label])
    
    # Anotar valores extremos
    for _, row in z_df.iterrows():
        if abs(row['Z-score']) > 1:
            ax.text(row['Z-score'] + 0.1 * np.sign(row['Z-score']), 
                   list(z_df['Feature']).index(row['Feature']),
                   f"{row['Z-score']:+.1f}σ", va='center', fontsize=8)

plt.suptitle('Análisis de predicciones individuales\n(rojo = por encima del promedio, azul = por debajo)',
             fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla detallada de cada caso
print("DETALLE DE PREDICCIONES INDIVIDUALES")
print("=" * 80)

for label, (idx, prob) in cases.items():
    case_values = X_risk_test.iloc[idx]
    actual = y_risk_test.iloc[idx]
    
    print(f"\n{'─' * 40}")
    print(f"Caso: {label}")
    print(f"P(agresivo) = {prob:.4f}")
    print(f"Clase real: {'Agresivo' if actual == 1 else 'No agresivo'}")
    print(f"Predicción correcta: {'Sí' if (prob >= 0.5) == actual else 'No'}")
    print(f"\nValores de features:")
    
    for feat in feat_risk:
        val = case_values[feat]
        mean = global_means[feat]
        std = global_stds[feat]
        z = (val - mean) / std if std > 0 else 0
        indicator = '▲' if z > 1 else '▼' if z < -1 else ' '
        print(f"  {indicator} {feat:<30s} = {val:>10.2f}  (promedio: {mean:.2f}, z={z:+.1f})")

print(f"\n{'─' * 40}")
print("\n→ ▲ indica valores significativamente por encima del promedio")
print("→ ▼ indica valores significativamente por debajo del promedio")
print("→ Esto permite explicar a un stakeholder POR QUÉ el modelo clasificó cada caso")

## 5. Model Cards

Una **Model Card** es un documento estandarizado que describe un modelo de ML.
Incluye: propósito, datos, métricas, limitaciones y consideraciones de equidad.

Son esenciales para la documentación y el uso responsable de los modelos.

In [ ]:
# Calcular métricas para las model cards
y_pred_risk_test = risk_model_obj.predict(X_risk_test)
y_prob_risk_test = risk_model_obj.predict_proba(X_risk_test)[:, 1]
f1_risk = f1_score(y_risk_test, y_pred_risk_test, zero_division=0)
auc_risk = roc_auc_score(y_risk_test, y_prob_risk_test)

y_pred_churn_test = churn_model_obj.predict(X_churn_test)
y_prob_churn_test = churn_model_obj.predict_proba(X_churn_test)[:, 1]
f1_churn_val = f1_score(y_churn_test, y_pred_churn_test, zero_division=0)
auc_churn = roc_auc_score(y_churn_test, y_prob_churn_test)

print(f"Riesgo - F1: {f1_risk:.4f}, AUC: {auc_risk:.4f}")
print(f"Churn  - F1: {f1_churn_val:.4f}, AUC: {auc_churn:.4f}")

In [ ]:
# Model Card 1: Clasificación de Riesgo de Conductor
model_card_risk = f"""
# MODEL CARD: Clasificación de Conductores de Alto Riesgo

## Propósito
Identificar conductores con comportamiento agresivo (alto riesgo) a partir de datos de telemetría vehicular.
Permite alertas proactivas, ajuste de primas de seguro y programas de capacitación dirigidos.

## Modelo
- **Tipo**: {risk_pkg['model_name']}
- **Features**: {len(feat_risk)} variables de telemetría
- **Target**: Binario (agresivo = top 20% en frenadas bruscas + variabilidad de velocidad)

## Datos
- **Fuente**: Datos de telemetría de la flota vehicular
- **Volumen**: {len(X_risk)} vehículos ({len(X_risk_train)} train / {len(X_risk_test)} test)
- **Balance**: {y_risk.mean()*100:.1f}% agresivos
- **Periodo**: Datos históricos de la flota

## Rendimiento (en set de test)
- **F1-Score (agresivo)**: {f1_risk:.4f}
- **AUC-ROC**: {auc_risk:.4f}
- **Recall (agresivo)**: {recall_score(y_risk_test, y_pred_risk_test, zero_division=0):.4f}
- **Precisión (agresivo)**: {precision_score(y_risk_test, y_pred_risk_test, zero_division=0):.4f}

## Features más importantes
{chr(10).join([f'- {f}' for f in perm_risk_df.tail(5)['Feature'].values[::-1]])}

## Limitaciones
- La definición de "agresivo" es una proxy basada en percentiles, no en accidentes reales
- El modelo no captura contexto situacional (ej. emergencias, condiciones climáticas)
- Entrenado con datos de una flota específica; puede no generalizar a otras flotas
- No considera cambios de comportamiento en el tiempo (solo snapshot)

## Consideraciones de equidad
- Verificar que el modelo no discrimine por tipo de vehículo de manera injusta
- Los conductores de vehículos deportivos pueden ser flaggeados más frecuentemente por características del vehículo, no por su comportamiento
- Se recomienda auditar las predicciones estratificadas por tipo de vehículo

## Uso recomendado
- Sistema de alertas para gestores de flota
- Input para cálculo de primas de seguro
- Selección de candidatos para programas de conducción segura
- NO usar como única base para decisiones punitivas
"""

display(Markdown(model_card_risk))

In [ ]:
# Model Card 2: Predicción de Churn
model_card_churn = f"""
# MODEL CARD: Predicción de Riesgo de Abandono (Churn)

## Propósito
Identificar clientes con alto riesgo de abandonar el servicio, permitiendo intervención proactiva
para mejorar la retención y reducir costos de adquisición.

## Modelo
- **Tipo**: {churn_pkg['model_name']}
- **Features**: {len(feat_churn)} variables (demográficas + telemetría + encuesta)
- **Target**: Binario (churn = satisfaction_score <= 2 AND would_recommend == 0)
- **Técnica de balanceo**: {'SMOTE' if HAS_SMOTE else 'Oversampling manual / scale_pos_weight'}

## Datos
- **Fuente**: Encuestas de compradores + datos de telemetría fusionados
- **Volumen**: {len(X_churn)} clientes ({len(X_churn_train)} train / {len(X_churn_test)} test)
- **Balance**: {y_churn.mean()*100:.1f}% churn (desbalance {'severo' if y_churn.mean() < 0.1 else 'moderado'})

## Rendimiento (en set de test)
- **F1-Score (churn)**: {f1_churn_val:.4f}
- **AUC-ROC**: {auc_churn:.4f}
- **Recall (churn)**: {recall_score(y_churn_test, y_pred_churn_test, zero_division=0):.4f}
- **Precisión (churn)**: {precision_score(y_churn_test, y_pred_churn_test, zero_division=0):.4f}

## Features más importantes
{chr(10).join([f'- {f}' for f in perm_churn_df.tail(5)['Feature'].values[::-1]])}

## Limitaciones
- El churn se define como proxy (satisfacción + no recomendación), no como abandono real observado
- Desbalance severo de clases puede afectar la confiabilidad de las predicciones
- No incluye datos competitivos (ofertas de la competencia, cambios de mercado)
- Las encuestas capturan un momento puntual; la satisfacción puede cambiar

## Consideraciones de equidad
- Verificar que el modelo no discrimine por edad, género o nivel socioeconómico
- Los clientes de income_bracket bajo pueden tener patrones de uso diferentes pero legítimos
- No usar el modelo para negar servicios, solo para mejorarlos

## Uso recomendado
- Identificar clientes para campañas de retención
- Priorizar contactos de soporte proactivo
- Personalizar ofertas de renovación
- Alimentar dashboards de gestión de clientes
"""

display(Markdown(model_card_churn))

In [ ]:
# Resumen visual: comparación de los dos modelos
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Métricas lado a lado
metrics_names = ['F1', 'AUC-ROC', 'Recall', 'Precisión']
risk_values = [
    f1_risk, auc_risk,
    recall_score(y_risk_test, y_pred_risk_test, zero_division=0),
    precision_score(y_risk_test, y_pred_risk_test, zero_division=0)
]
churn_values = [
    f1_churn_val, auc_churn,
    recall_score(y_churn_test, y_pred_churn_test, zero_division=0),
    precision_score(y_churn_test, y_pred_churn_test, zero_division=0)
]

x = np.arange(len(metrics_names))
width = 0.35

ax = axes[0]
ax.bar(x - width/2, risk_values, width, label='Riesgo conductor', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, churn_values, width, label='Churn', color='#9b59b6', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylabel('Score')
ax.set_title('Comparación de Rendimiento', fontsize=14, fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.1)

# Anotar valores
for i, (rv, cv) in enumerate(zip(risk_values, churn_values)):
    ax.text(i - width/2, rv + 0.02, f'{rv:.3f}', ha='center', fontsize=9)
    ax.text(i + width/2, cv + 0.02, f'{cv:.3f}', ha='center', fontsize=9)

# Resumen en tabla
ax = axes[1]
table_data = [
    ['Modelo', 'Riesgo Conductor', 'Churn'],
    ['Tipo', risk_pkg['model_name'][:20], churn_pkg['model_name'][:20]],
    ['Features', str(len(feat_risk)), str(len(feat_churn))],
    ['Balance', f"{y_risk.mean()*100:.1f}% pos", f"{y_churn.mean()*100:.1f}% pos"],
    ['F1', f"{f1_risk:.4f}", f"{f1_churn_val:.4f}"],
    ['AUC-ROC', f"{auc_risk:.4f}", f"{auc_churn:.4f}"],
]

table = ax.table(cellText=[r[1:] for r in table_data[1:]],
                 rowLabels=[r[0] for r in table_data[1:]],
                 colLabels=table_data[0][1:],
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2)
ax.axis('off')
ax.set_title('Resumen de Modelos', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## Resumen: Fase 4 Completa

### Lo que construimos en la Fase 4

| Notebook | Modelo | Objetivo | Técnicas clave |
|---|---|---|---|
| 03 | Regresión de Satisfacción | Predecir satisfaction_score | Regresión Lineal, Ridge, RF, XGBoost |
| 04 | Clasificación de Riesgo | Identificar conductores agresivos | Logística, RF+GridSearch, XGBoost, umbral |
| 05 | Clasificación de Churn | Detectar clientes en riesgo de irse | SMOTE, scale_pos_weight, RF, XGBoost |
| 06 | Interpretación | Explicar las predicciones | Permutation importance, PDPs, Model Cards |

### Valor de negocio generado
- → **Seguridad vial**: sistema de alertas para conductores de alto riesgo
- → **Retención de clientes**: detección temprana de churn con intervención proactiva
- → **Satisfacción del cliente**: entendimiento de qué factores impulsan la satisfacción
- → **Transparencia**: model cards y técnicas de interpretación para comunicar al negocio

### Aprendizajes técnicos
1. La **selección de métricas** depende del contexto de negocio (no siempre accuracy)
2. El **desbalance de clases** requiere técnicas específicas (SMOTE, pesos, muestreo)
3. El **ajuste de umbral** puede mejorar significativamente el valor práctico del modelo
4. La **interpretación** es tan importante como el rendimiento del modelo
5. Las **model cards** son esenciales para el uso responsable de ML

### Siguiente: Fase 5
→ Aprendizaje No Supervisado: clustering, reducción de dimensionalidad, detección de anomalías